# Exercise 1: Testing Transactions and Cursor Queries in PostgreSQL

While the lesson provided an overview of what ACID transactions were, and the types of gaurantees provided by relational databases, in this exercise you'll be able to explore through a hands on example, and write some of those transactions yourself, while you commit them to the database via the connection you have setup to PostgreSQL. As you fill in the blanks for the code below, you'll be able to test ACID guarantees and demonstrate the link between constraints and transaction behavior.

## Step 1: Preparing Tables & Data
Before we can test transactions via a Data Markup Language (i.e. INSERT, UPDATE, etc.), we must define our tables and their "rules" using DDL. These rules are what the Consistency guarantee will enforce. 

In [2]:
%%capture
%pip install notebook duckdb==1.3 jupysql matplotlib duckdb-engine psycopg2;
%config SqlMagic.autopandas = True;
%config SqlMagic.feedback = False;
%config SqlMagic.displaycon = False;
import duckdb;
import pandas as pd
import zipfile;
import os;
from io import StringIO
import psycopg2
from psycopg2 import Error

In [36]:
def connect():
    # 1. Establish the connection
    try:
        # Connect to the PostgreSQL database
        conn = psycopg2.connect(
            dbname='postgres',
            user='postgres',
            password='I4NAennDnRh1uO9z2y0k6pMfAf',
            host='localhost',
            port='5432')

        cursor = conn.cursor()
        print("Connection successful.")
        return cursor, conn
    
    except psycopg2.Error as e:
        # Handle any errors during connection or query execution
        print(f"\nAn error occurred: {e}")

def execute_query(cursor, conn, sql_query, fetch = False, commit = True):
    cursor.execute(sql_query)
    if fetch == True:
        results = cursor.fetchall()
        return results
    if commit == True:
        conn.commit()

### First: A Review of Transactions

Recall that transactions are "bundles" of operations. As per PostgreSQL documentation: "The essential point of a transaction is that it bundles multiple steps into a single, all-or-nothing operation. The intermediate states between the steps are not visible to other concurrent transactions, and if some failure occurs that prevents the transaction from completing, then none of the steps affect the database at all". We can specify what queries and SQL operations are a part of a single transaction using the commands:

- BEGIN;
- COMMIT; 

Note that if anything goes wrong, we can use the commands:

- SAVEPOINT my_savepoint;
- ROLLBACK TO my_savepoint;
- ROLLBACK

To: 

- Create a sort of "checkpoint" which will allow us to "save" a portion of a transaction, and commit everything until the savepoint, while discarding the rest. 
- Rollback to a given savepoint (in the event something goes wrong with the rest of the transaction), where we can then decide what to do from that point (commit everything up until the savepoint or discard everything alltogether) 
- Rollback the entire transaction, including everything done before and after the savepoint

Let's go ahead and try a single transaction with only one sql command in it: a create table statement for our customers table.

In [ ]:
sql_query = """
BEGIN;
CREATE TABLE Customers (
    customer_id INT PRIMARY KEY,
    full_name   VARCHAR(100) NOT NULL,
    email       VARCHAR(100) NOT NULL UNIQUE
);
COMMIT; 
"""

connection = connect()
execute_query(connection[0], connection[1], sql_query, fetch = False)


Next, let's try to create accounts and balances for these customers. We'll also add data, this time in the same transaction.

In [15]:
sql_query = """
BEGIN;

/*
 * A table for bank accounts.
 * We want to ensure that an account balance can NEVER be negative.
 * We also link each account to a valid customer.
 */

CREATE TABLE Accounts (
    account_id  INT PRIMARY KEY,
    customer_id INT NOT NULL,
    balance     DECIMAL(10, 2) NOT NULL CHECK (balance >= 0),
    
    -- This is the "link" that ensures an account belongs to a real customer
    CONSTRAINT fk_customer
        FOREIGN KEY(customer_id) 
        REFERENCES Customers(customer_id)
);

-- Let's add some starting data
INSERT INTO Customers (customer_id, full_name, email) VALUES
(1, 'Alice Smith', 'alice@example.com'),
(2, 'Bob Johnson', 'bob@example.com');

INSERT INTO Accounts (account_id, customer_id, balance) VALUES
(101, 1, 1000.00),  -- Alice's account
(102, 2, 250.00);   -- Bob's account

COMMIT; 
"""

connection = connect()
execute_query(connection[0], connection[1], sql_query, fetch = False)


Connection successful.


DuplicateTable: relation "accounts" already exists


Notice how we can run several SQL commands - including DDL, which defines the new account balances table, and DML, which adds extra data to customers and accounts and inserts records, in a single transaction.

Let's go ahead and try to test some of our consistency rules. Notice how we set specific contstraints:

- account_id, which is our primary key for the table of accounts, cannot be null - neither can customer_id (as you must be a customer to have an account with the company)
- the balance of each customer cannot be null - it must either be zero, or it must be some decimal value. We also do not allow fractions of a cent (as we do not have currency in those units)
- referencing the original customer table DDL creation script, customers must have names and emails. Those emails must be unique (as of course, customers cannot copy each others emails - each email has to be unique to each person)

Let's test how well our constraints, defined in the DDL here, hold up!

### Testing 'C' (Consistency)
Let's see how the DDL constraints (NOT NULL, UNIQUE, CHECK, FOREIGN KEY) immediately reject invalid data, enforcing the "valid state" of the database. These will all be single-statement "transactions" that fail.

Try to create a new customer without an email address.

In [16]:
sql_query = """
BEGIN;
INSERT INTO Customers (customer_id, full_name) 
VALUES (3, 'Charlie Brown');
"""

connection = connect()
execute_query(connection[0], connection[1], sql_query, fetch = False)


Connection successful.


NotNullViolation: null value in column "email" of relation "customers" violates not-null constraint
DETAIL:  Failing row contains (3, Charlie Brown, null).


Notice that the transaction has failed - even though we didn't commit it yet. We have an error because the null value is unacceptable based on the rules we've implemented - the email field is not "nullable".

We should roll back this operation - in order to prevent this error from impacting the state of the database, and having this incorrect transaction linger. 

In [19]:
sql_query = """
ROLLBACK;
"""
execute_query(connection[0], connection[1], sql_query, fetch = False)


We've learned that our initial DDL commands that defined the table properties (i.e. field - NOT NULL) protected the database's consistency.

Let's now test the UNIQUE Constraint. We'll try to create a new customer using an existing email.

In [18]:
sql_query = """
BEGIN;
INSERT INTO Customers (customer_id, full_name, email) 
VALUES (4, 'Eve Davis', 'alice@example.com');
"""
connection = connect()
execute_query(connection[0], connection[1], sql_query, fetch = False)

Connection successful.


UniqueViolation: duplicate key value violates unique constraint "customers_email_key"
DETAIL:  Key (email)=(alice@example.com) already exists.


As expected: the query failed once again. The database will reject this insert statement operation with an error about duplicate key values violating unique constraints against the "customers_email_key". The column UNIQUE constraints ensured no two customers can have the same email. Once again, let's rollback the operation.



In [21]:
sql_query = """
ROLLBACK;
"""
execute_query(connection[0], connection[1], sql_query, fetch = False)

Next, let's try to test constraints against the values of the customer's balance. We will try to update Alice's balance to be a negative number.

In [22]:
sql_query = """
BEGIN;
UPDATE Accounts 
SET balance = -50.00 
WHERE account_id = 101;
"""
connection = connect()
execute_query(connection[0], connection[1], sql_query, fetch = False)

Connection successful.


CheckViolation: new row for relation "accounts" violates check constraint "accounts_balance_check"
DETAIL:  Failing row contains (101, 1, -50.00).


As expected, the query fails. The database rejects this with error about the new row for relation "accounts" violating check constraint "accounts_balance_check". Once again, we can rollback this adjustment.

The CHECK (balance >= 0) protected the business rule that accounts cannot be overdrawn.


In [25]:
sql_query = """
ROLLBACK;
"""
execute_query(connection[0], connection[1], sql_query, fetch = False)

Let's now test the FOREIGN KEY (Referential) Constraint. Try to create a new account for a customer ID that doesn't exist. This shouldn't be possible, because again - only customers should have accounts with us.

In [26]:
sql_query = """
BEGIN;
INSERT INTO Accounts (account_id, customer_id, balance) 
VALUES (103, 999, 50.00);
"""

connection = connect()
execute_query(connection[0], connection[1], sql_query, fetch = False)

Connection successful.


ForeignKeyViolation: insert or update on table "accounts" violates foreign key constraint "fk_customer"
DETAIL:  Key (customer_id)=(999) is not present in table "customers".


Expected Result: Failure. The database will reject this with an error like: ERROR: insert or update on table "accounts" violates foreign key constraint "fk_customer".

The lesson here is that the FOREIGN KEY constraint ensured that every account is properly linked to a real, existing customer.

### Part  2: Testing 'A' (Atomicity)
We now want to see and ensure that a transaction is "all or nothing." We will simulate a bank transfer. A transfer involves two steps: subtracting money from one account and adding it to another. Both must succeed, or neither should happen.

Let's start with the "Happy Path" - what happens when a transaction goes well. Let's transfer $100 from Alice (101) to Bob (102).

In [27]:
sql_query = """
BEGIN TRANSACTION;

-- Step 1: Subtract $100 from Alice
UPDATE Accounts SET balance = balance - 100.00 WHERE account_id = 101;

-- Step 2: Add $100 to Bob
UPDATE Accounts SET balance = balance + 100.00 WHERE account_id = 102;

COMMIT; -- Finalize the transaction
"""

connection = connect()
execute_query(connection[0], connection[1], sql_query, fetch = False)

Connection successful.


In [ ]:
Expected Result: ✅ Success. Alice will have $900.00 and Bob will have $350.00. Both updates worked, so the COMMIT made them permanent.

In [28]:
sql_query = """
SELECT * FROM Accounts;
"""

connection = connect()
execute_query(connection[0], connection[1], sql_query, fetch = True)


Connection successful.


[(101, 1, Decimal('900.00')), (102, 2, Decimal('350.00'))]

Expected Result: ✅ Success. Alice will have 900.00 dollars and Bob will have 350.00. Both updates worked, so the COMMIT made them permanent.

Let's move onto the "Failure Path" i.e. an Atomic Rollback. Start with a transfer $5,000 from Alice to Bob. Alice only has $900. The CHECK constraint will stop this.

In [29]:
sql_query = """
BEGIN TRANSACTION;

-- Step 1: Subtract $5000 from Alice (This will fail the CHECK constraint)
UPDATE Accounts SET balance = balance - 5000.00 WHERE account_id = 101;

-- Step 2: Add $5000 to Bob (This line may not even be reached)
UPDATE Accounts SET balance = balance + 5000.00 WHERE account_id = 102;

COMMIT; -- Try to finalize
"""

connection = connect()
execute_query(connection[0], connection[1], sql_query, fetch = True)

Connection successful.


CheckViolation: new row for relation "accounts" violates check constraint "accounts_balance_check"
DETAIL:  Failing row contains (101, 1, -4100.00).


Expected Result: 🛑 Failure. The database will throw a CHECK constraint error (from Exercise 1C) when it tries to execute the first UPDATE. The COMMIT will fail and/or you'll be forced to ROLLBACK. 
In general, we will ROLLBACK regardless, as it's good practice to do so after a transaction has failed. 

Let's see if this failed correctly - i.e. that both transactions failed, not just one. 

In [30]:
sql_query = """
ROLLBACK; -- (Good practice to explicitly rollback after an error)
SELECT * FROM Accounts;
"""

execute_query(connection[0], connection[1], sql_query, fetch = True)

[(101, 1, Decimal('900.00')), (102, 2, Decimal('350.00'))]

You will see that Alice still has 900.00 dollars and Bob still has 350.00. Crucially, Bob did not get $5000, and Alice's account was not set to a negative value. Because one part of the transaction failed (Step 1), the entire transaction was aborted. This is Atomicity.

### Part 3: Testing 'I' (Isolation)
We've shown the database can gaurantee Atomicity, and Consistency. Next, we want to show that one transaction doesn't see the "dirty" or uncommitted work of another. This requires two separate database connections - which we can simulate here with the help of our Postgres driver.

Here's the starting Point: Alice (101) has $900.00.

In Session 1 (Your "Updater" Window) we will:
1. Run BEGIN TRANSACTION;	
2. Run UPDATE Accounts SET balance = 50.00 WHERE account_id = 101; (Alice's new balance is $50, but don't COMMIT!)	
3. Check the balance: SELECT balance FROM Accounts WHERE account_id = 101; 

You should and you will see the balance is 50.00 dollars.

NEXT - 

4. Now, in this separate window - Session 2 (Your "Reader" Window), check the balance: SELECT balance FROM Accounts WHERE account_id = 101;
Expected Result: Session 2 sees **900.00** dollars. It does not see the uncommitted 50.00 dollar value. It is isolated from Session 1's incomplete work.
5. Now, back in Session 1, run: COMMIT;	
6. Immediately after Session 1 commits, run this again in Session 2: SELECT balance FROM Accounts WHERE account_id = 101;
Expected Result: Now Session 2 sees **$50.00**. It can only see the new value after it has been fully and durably committed.


As you can see, isolation prevents other connections from reading "dirty" or "in-progress" data, which would lead to massive confusion and inconsistency.

In [37]:
sql_query = """
BEGIN TRANSACTION;
UPDATE Accounts SET balance = 50.00 WHERE account_id = 101; 
SELECT balance FROM Accounts WHERE account_id = 101;
"""

connection_one = connect()
execute_query(connection_one[0], connection_one[1], sql_query, fetch = True, commit = False)

Connection successful.


[(Decimal('50.00'),)]

In [38]:
sql_query = """
SELECT balance FROM Accounts WHERE account_id = 101;
"""

connection_two = connect()
execute_query(connection_two[0], connection_two[1], sql_query, fetch = True, commit = False)

Connection successful.


[(Decimal('900.00'),)]

In [40]:
sql_query = """
BEGIN TRANSACTION;
UPDATE Accounts SET balance = 50.00 WHERE account_id = 101; 
COMMIT;
"""
execute_query(connection_one[0], connection_one[1], sql_query, fetch = False, commit = True)

In [41]:
sql_query = """
SELECT balance FROM Accounts WHERE account_id = 101;
"""

connection_two = connect()
execute_query(connection_two[0], connection_two[1], sql_query, fetch = True, commit = False)

Connection successful.


[(Decimal('50.00'),)]

Which is exactly what we expected! The isolation property works perfectly.

So far, we have the ACI - Atomicity, Consistency, and Isolation gaurantees of the database tested. Let's add the D in ACID. 

### Part 4: Testing 'D' (Durability)
We need to show that once COMMIT is executed, the data is safe and permanent. In most terms, this usually means it's stored on disk and not volatile memory. Let's give Bob a $77.00 bonus

In [ ]:
sql_query = """
BEGIN;
UPDATE Accounts SET balance = balance + 77.00 WHERE account_id = 102;
COMMIT;
"""

connection = connect()
execute_query(connection[0], connection[1], sql_query, fetch = False, commit = True)

Let's make sure the data is all there. We should see Bob's new balance, with his increased bonus.


In [46]:
sql_query = """
SELECT balance FROM Accounts WHERE account_id = 102;
"""

connection = connect()
execute_query(connection[0], connection[1], sql_query, fetch = True, commit = False)

Connection successful.


[(Decimal('504.00'),)]

In [ ]:
Here comes the test: Now, completely close your database connection. Shut down the query tool. If you could, you'd reboot the whole server. Then, reconnect: Open a brand new connection to the database. Since we are running in a lab environment, we can't shut down everything - so we'll just create a new connection after closing the old one, and run the check again:

In [48]:
sql_query = """
SELECT balance FROM Accounts WHERE account_id = 102;
"""

connection = connect()
execute_query(connection[0], connection[1], sql_query, fetch = True, commit = False)

Connection successful.


[(Decimal('504.00'),)]

Lesson: The value is still the same. Because the transaction was COMMITted, the database guarantees that this change is Durable and has survived the "crash" (i.e., you disconnecting). This is the opposite of ROLLBACK.